# 11 — A deep agent, end to end

**What you'll learn**

- What a "deep agent" is — a long-horizon loop that plans, uses tools, delegates to sub-agents, and stays inside guardrails — and that you have already built every part of one
- How to assemble one from Parts 1-3 by composition: `run_agent` + a `run_subagent` policy firewall + a `require_approval` gate wired to the `rules.decide` verifier + a `Budget` ceiling + a `ContextLedger`, all in a few cells
- Clearing a small morning queue of dev tickets with it, then grading the results with `score_ticket` and `check_decision` against gold
- The same brief expressed in LangChain's `deepagents` (`create_deep_agent`) — a framework that *packages* the same machinery as planning, filesystem, and sub-agent middleware
- The honest trade: what `deepagents` hands you for free (assembled, maintained middleware) versus the transparency and control you gave up to get it

*Time: ~4 min on a first live run; under a minute cached. Cost: ~$0.03 (the `deepagents` section makes a handful of uncached calls). Cached reruns are free.*

> **Before running this notebook:** `pip install -e ".[langgraph]"` (once). It pulls in `langgraph`, `langchain`, and `deepagents` — the framework we translate the brief into at the end. Everything else stays the same.

## The brief: clear the morning queue

Ten chapters in, the ops desk has a drawer full of parts. Chapter 02 built the tool-calling loop; chapter 04 gave it a score against gold; chapter 06 added a deterministic decision check; chapter 08 wrapped risky tools in an approval gate and put a `Budget` on the run; chapter 10 taught it to move bulk out of context and to delegate work to a sub-agent that never pollutes the parent's window. Each was a lesson in isolation. This chapter spends them all at once.

The brief is the daily reality of the desk, stated as one job: **clear the morning queue.** A handful of return tickets have come in overnight. For each, the agent must look up the order and the customer, find the governing policy, compute the amount, carry out the decision — and it must do so *within a budget*, *never moving money a rule would not*, and *leaving a trace we can read afterward*.

That word "and" is the whole chapter: a capable agent is not one clever trick but several plain ones, composed. We build ours by hand first — every seam visible — then rebuild the exact same brief in `deepagents`, a framework whose value is precisely that it has already assembled these seams for you. Seeing both back to back is the point: a framework is a *packaging* of the machinery you now understand, not a different kind of magic.

Anthropic's own guidance frames the discipline as [context engineering](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents) — curating the tokens an agent sees — and the [orchestrator-workers pattern](https://www.anthropic.com/engineering/multi-agent-research-system) of a lead agent delegating to focused sub-agents; both are things we wire up below with parts we already own.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

Every model call our hand-built agent makes still routes through `shoplab.llm.complete`, so Phoenix traces the whole morning queue — parent turns, sub-agent policy lookups, gate decisions — in one project. Reading the trace afterward is how you tell a good run from a lucky one. The `deepagents` section calls the model through its own client, so those turns show up under LangChain's instrumentation instead; the same trace view, a different producer. Optional as always: skip it and nothing else changes.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## The queue, and the answer key

The queue is four dev tickets, chosen to span the decisions a desk actually makes: a clean refund, a partial refund with a restocking fee, a denial past the return window, and a warranty replacement. The gold labels beside them are the answer key `shoplab.rules.decide` computed in chapter 04 — the agent never sees them; we grade against them at the end. Two of the four (`approve_refund`, `partial_refund`) move real money, which is exactly where the gate will earn its keep.

In [ ]:
from shoplab.world import load_tickets, load_orders, load_customers

orders = {o["order_id"]: o for o in load_orders()}
customers = {c["customer_id"]: c for c in load_customers()}
tickets = {t["ticket_id"]: t for split in load_tickets().values() for t in split}

def render(t):                                    # the ticket as the agent will read it
    return (f"Ticket {t['ticket_id']} from {t['customer_id']} about order "
            f"{t['order_id']}, sku {t['sku']} qty {t['qty']}, condition "
            f"{t['item_condition']}, {t['days_since_delivery']} days since delivery, "
            f"photo evidence {t['evidence_photo']}, requested {t['requested_action']}. "
            f"Customer writes: {t['reason_text']}")

QUEUE = ["TKT-2224", "TKT-2225", "TKT-2226", "TKT-2231"]
for tid in QUEUE:
    g = tickets[tid]["gold"]
    print(f"{tid}  {tickets[tid]['requested_action']:12}{tickets[tid]['item_condition']:10}"
          f"-> {g['decision']:15} {g['policy_id']:16} ${g['refund_usd']}")

> **What you should see:** four tickets whose gold decisions are `approve_refund`, `partial_refund`, `deny`, and `replacement` — one of each shape the desk handles, and two that move money. This spread is deliberate: a queue of four identical refunds would prove nothing. Keep the gold column in view; every number the agent produces below gets checked against it.

## Piece one: a policy sub-agent as a context firewall

Start with the part chapter 10 built last. Policy documents are long, and a naive agent that calls `search_policy` inline drags the full text of every hit into its own context — where it sits for the rest of the run, crowding the window and inviting the model to re-read it. `run_subagent` is the fix: it runs a *fresh* loop with its own tiny toolset and hands the parent back only the answer, never the intermediate turns. That is the firewall — the parent asks a question and gets one sentence, while the policy texts, the searches, and the sub-agent's deliberation stay quarantined on the other side.

We wrap that primitive as a single tool, `research_policy`, that the desk agent can call like any other. Its sub-agent holds exactly one tool — `search_policy` — and is told to look once and report back in a line. The demo call below prints one such finding: an opened in-window return names `pol-restocking` and its 10% fee, and everything the sub-agent read to get there stays on its side of the wall.

In [ ]:
from shoplab.tools import standard_tools, Tool
from shoplab.context import run_subagent

RESEARCH_SYSTEM = ("You are a policy researcher for Larkspur. Call search_policy ONCE for the "
    "question, then STOP calling tools and reply in one plain sentence: name the governing "
    "policy id (like pol-restocking) and the rule it states. Do not decide the ticket.")

def research_policy_tool(model):
    def research_policy(question):
        sub = run_subagent(question, {"search_policy": standard_tools()["search_policy"]},
                           model=model, max_steps=4, system=RESEARCH_SYSTEM)
        return {"finding": sub["answer"] or "no policy text returned; try search_policy again"}
    return Tool("research_policy",
                "Ask a policy sub-agent a question; returns a one-line policy finding.",
                {"type": "object", "properties": {"question": {"type": "string"}},
                 "required": ["question"]}, research_policy)

demo = research_policy_tool(MODEL).fn(question="refund for an opened item returned in 20 days")
print(demo["finding"])

## Piece two: an approval gate wired to the verifier

Chapter 08's `require_approval` makes a risky tool wait for a yes; chapter 06's `check_decision` recomputes the gold decision from `rules.decide` with no model call. Compose them and you get the guardrail a real desk needs: the *approver itself* consults the rules, so a refund executes only when the deterministic engine agrees on both the decision and the amount, to the cent. The model can propose whatever it likes; money moves only when policy backs the exact figure.

`rules_approver` closes over one ticket's order and customer, computes the gold once, and returns an `(name, args) -> bool` approver of the shape `require_approval` expects.

In [ ]:
from shoplab.rules import decide
from shoplab.controls import require_approval

def rules_approver(ticket, order, customer):
    gold = decide(ticket, order, customer)                 # the deterministic answer key
    def approve(name, args):
        if name == "issue_refund":
            return (gold["decision"] in ("approve_refund", "partial_refund")
                    and abs(args.get("amount_usd", 0) - (gold["refund_usd"] or 0)) <= 0.01)
        if name == "create_replacement":
            return gold["decision"] == "replacement"
        return True
    return approve

t = tickets["TKT-2224"]                                     # gold: approve_refund, $36.50
gate = rules_approver(t, orders[t["order_id"]], customers[t["customer_id"]])
print("wrong amount ($42.45):", gate("issue_refund", {"amount_usd": 42.45}))
print("right amount ($36.50):", gate("issue_refund", {"amount_usd": 36.50}))

The cell above judged the same call `False` at \$42.45 and `True` at \$36.50: the gate refuses to let money move unless the rules engine independently arrives at the same amount, so an over-generous refund is stopped at the door and the agent must retry at the figure policy supports.

Now the composition. `clear_ticket` builds one ticket's toolset from parts: the read-only lookups and `finish` come straight from `standard_tools`, `research_policy` is our sub-agent firewall, and the two risky tools are wrapped in that verified gate — all sharing one `Ledger` so every side effect lands in one place. The loop itself is chapter 02's `run_agent`; a `Budget` on its `on_step` hook caps the run at 16 calls and 5 cents, tripping the instant either ceiling is crossed. That is the entire agent: a plain loop, a scoped toolset, a firewall, a gate, and a budget.

Below we define it, then run it over the whole queue — each ticket gets a fresh toolset, ledger, and budget, so nothing leaks between them.

In [ ]:
import shoplab.llm
from shoplab.loop import run_agent
from shoplab.tools import Ledger
from shoplab.controls import Budget

READ = ("get_order", "get_customer", "calc", "escalate", "finish")
DESK_SYSTEM = ("You are the Larkspur Outfitters ops desk. Clear the ticket: look up the order "
    "and the customer, call research_policy once for the governing rule, compute any amount "
    "with calc, then carry out the decision -- issue_refund when money is due, create_replacement "
    "for a replacement, escalate for human review -- and only then call finish with decision, "
    "policy_id, and refund_usd (number or null). Policy, not sympathy.")

def clear_ticket(ticket, model=MODEL):
    order, customer = orders[ticket["order_id"]], customers[ticket["customer_id"]]
    ledger, budget = Ledger(), Budget(max_calls=16, max_cost_usd=0.05)
    base = standard_tools(ledger)
    tools = {n: base[n] for n in READ}
    tools["research_policy"] = research_policy_tool(model)
    approve = rules_approver(ticket, order, customer)
    for name in ("issue_refund", "create_replacement"):
        tools[name] = require_approval(base[name], approve)
    r = run_agent(render(ticket), tools, system=DESK_SYSTEM, max_steps=14,
                  on_step=lambda step, msg: budget.charge(shoplab.llm.LEDGER[-1]["cost_usd"]))
    return {"ticket": ticket, "answer": r.answer, "messages": r.messages,
            "moved": ledger.entries, "budget": budget.snapshot(), "stop": r.stop_reason}

In [ ]:
runs = [clear_ticket(tickets[tid]) for tid in QUEUE]

for run in runs:
    ans = run["answer"] if isinstance(run["answer"], dict) else {"decision": run["answer"]}
    print(f"{run['ticket']['ticket_id']}  stop={run['stop']:9} "
          f"decision={str(ans.get('decision')):15} "
          f"moved={[e['kind'] for e in run['moved']] or '[]'}  "
          f"calls={run['budget']['calls']}")

> **What you should see:** four runs that each end at `finish`, inside the 16-call budget. One refund ticket runs noticeably longer than the others — here the agent re-queries the policy sub-agent several times before it commits. The two money tickets record exactly one `issue_refund` each, and what lands in the ledger is the policy-backed figure; the replacement records a `create_replacement`, and the denial records nothing. Temperature 0 is not determinism, so step counts wander run to run; that every ticket resolves inside budget is the invariant.

## Grading the queue

A run that finishes is not a run that is right. Bring back both verifiers from earlier chapters: `score_ticket` (chapter 04) compares the decision, policy, and amount to gold field by field, and `check_decision` (chapter 06) independently recomputes the gold and reports whether the agent agrees. One grades, the other audits; together they turn a run that merely finished into one you can trust.

In [ ]:
from shoplab.evals import score_ticket
from shoplab.verify import check_decision

print(f"{'ticket':9}{'decision':16}{'gold':16}{'exact':7}{'verified':10}{'moved $':>9}{'calls':>7}")
for run in runs:
    t = run["ticket"]
    pred = run["answer"] if isinstance(run["answer"], dict) else {}
    s = score_ticket(pred, t["gold"])
    chk = check_decision(pred, t, orders[t["order_id"]], customers[t["customer_id"]])
    moved = sum(e.get("amount_usd", 0) for e in run["moved"])
    print(f"{t['ticket_id']:9}{str(pred.get('decision')):16}{t['gold']['decision']:16}"
          f"{str(s['exact']):7}{str(chk['agrees']):10}{moved:>9.2f}{run['budget']['calls']:>7}")

> **What you should see:** the agent's decision matching gold on every row, `exact` and `verified` both `True`, and the money moved equal to the gold amount for the two refunds and \$0.00 for the denial and replacement. `exact` (from `score_ticket`) and `verified` (from `check_decision`) agree here because both consult the same rules engine — but they are different guarantees: one is a grade you compute offline against a labeled set, the other a check you could run in production on a ticket that has no label at all. A capable agent is one whose answers survive both.

One claim remains to make good on: the sub-agent firewall. If `research_policy` did its job, the parent's context never held a full policy document — only the one-line findings. `ContextLedger` (chapter 10) replays a finished transcript prefix by prefix to chart how the window grew; we read the peak, then search the parent's messages for a chunk of policy text that only ever lived inside the sub-agent.

In [ ]:
from shoplab.world import load_policies
from shoplab.context import ContextLedger

run = next(r for r in runs if r["ticket"]["ticket_id"] == "TKT-2225")   # a partial_refund
led = ContextLedger(model=MODEL)
for i in range(1, len(run["messages"]) + 1):
    led.observe(run["messages"][:i], label=run["messages"][i - 1]["role"])

policies = {p["id"]: p for p in load_policies()}
snippet = policies["pol-restocking"]["text"][:60]              # text seen only by the sub-agent
leaked = any(snippet in str(m.get("content", "")) for m in run["messages"])
print("parent messages:", len(run["messages"]), "| peak tokens:", led.peak())
print("full policy text ever in the parent context:", leaked)

> **What you should see:** a modest peak — on the order of a thousand tokens for the whole triage — and `False`: the full `pol-restocking` text never appears in the parent's messages. The parent knew the rule (its sub-agent told it in a sentence) without ever holding the document. That is the firewall paying off: delegation is not just tidier, it is *cheaper context*, which over a long queue is the difference between a run that stays coherent and one that drowns in its own transcript.

## The same brief in `deepagents`

Everything above is a few dozen lines you can read end to end. A framework's pitch is that you should not have to write them. LangChain's [`deepagents`](https://docs.langchain.com/oss/python/deepagents/overview) is a harness with "built-in capabilities for file systems for context management, subagent-spawning, and long-term memory" — precisely the pieces we just hand-wired. `create_deep_agent` returns a compiled [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) graph with planning, filesystem, sub-agent, and summarization middleware already stacked.

We express the same brief in it, cheaply — one ticket. Two adjustments from our stack: the model is a LangChain `ChatOpenAI` pointed at OpenRouter's OpenAI-compatible endpoint (note the model id drops the `openrouter/` prefix — that prefix is LiteLLM's routing syntax, not the API's), and our tools become plain annotated Python callables, which `deepagents` accepts directly.

In [ ]:
from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent
from shoplab.world import search_policy

da_model = ChatOpenAI(base_url="https://openrouter.ai/api/v1",
                      api_key=os.environ["OPENROUTER_API_KEY"],
                      model="deepseek/deepseek-v3.2", temperature=0)

def get_order(order_id: str) -> dict:
    """Look up a Larkspur order by id (items, totals, status, dates)."""
    return orders.get(order_id, {"error": f"no such order {order_id}"})

def get_customer(customer_id: str) -> dict:
    """Look up a Larkspur customer by id (tier, flags, history)."""
    return customers.get(customer_id, {"error": f"no such customer {customer_id}"})

def find_policy(query: str, k: int = 2) -> list:
    """Keyword-search the 12 Larkspur store policy documents."""
    return search_policy(query, k=k)

agent = create_deep_agent(model=da_model, tools=[get_order, get_customer, find_policy],
    system_prompt=("You are the Larkspur ops desk. Triage the return ticket: look up the order "
        "and customer, find the governing policy, then state the decision (approve_refund / "
        "partial_refund / replacement / store_credit / deny / escalate), the policy id, and the "
        "dollar amount. Be terse."))
print("agent is a:", type(agent).__module__ + "." + type(agent).__name__)

In [ ]:
ticket_text = render(tickets["TKT-2205"])          # gold: partial_refund, pol-restocking, $170.99
result = agent.invoke({"messages": [{"role": "user", "content": ticket_text}]})

msgs = result["messages"]
calls = [tc["name"] for m in msgs for tc in getattr(m, "tool_calls", []) or []]
print("state channels:", sorted(result))          # deepagents' own graph state
print("tool calls:", calls)
print("messages in the graph run:", len(msgs))
print("\nfinal answer (tail):\n" + msgs[-1].content[-450:])

> **What you should see:** `create_deep_agent` returned a `langgraph...CompiledStateGraph` — a graph, not a while-loop — and running it lands on the same decision our agent did: `partial_refund` under `pol-restocking` for \$170.99. The `state channels` include `messages` and `files`: that `files` key is the FilesystemMiddleware's virtual filesystem, deepagents' answer to context offload, present whether or not this run used it. Same brief, same answer, machinery you did not have to write. (The model here talks to OpenRouter through LangChain's client, so these calls are not on the LiteLLM disk cache — this one cell costs a fraction of a cent on every run.)

## Machinery map: what we built, where `deepagents` keeps it

Line the two up and the framework stops looking like magic. Every piece we composed by hand has a home in `deepagents` — usually a middleware, occasionally a graph feature. The differences are about *packaging and defaults*, not ideas.

| Our hand-built piece | Where it lives in `deepagents` |
|---|---|
| `run_agent` while-loop (`shoplab.loop`) | the compiled LangGraph graph — a `model` node and a `tools` node |
| `Tool` + `to_openai_tools` wiring | plain callables passed to `create_deep_agent(tools=...)`, bound by the graph |
| ad-hoc plan in the system prompt ("look up X, then Y") | `TodoListMiddleware` and its `write_todos` tool; the plan lives in a `todos` state channel |
| `offload` / `load_offload` blob handles (`shoplab.context`) | `FilesystemMiddleware` (`ls`, `read_file`, `write_file`, `edit_file`); bytes live in the `files` channel |
| `ContextLedger` + `compact` (`shoplab.context`) | `SummarizationMiddleware`, folding the transcript automatically as it grows |
| `run_subagent` firewall (`shoplab.context`) | `SubAgentMiddleware` and the `task` tool; sub-agents declared via `subagents=[...]` |
| `require_approval` gate (`shoplab.controls`) | `HumanInTheLoopMiddleware`, configured with `interrupt_on={...}` |
| `Budget` on `on_step` (`shoplab.controls`) | no built-in — you add a custom `AgentMiddleware` in the `middleware=[...]` slot |
| `Checkpoint` save/load (`shoplab.controls`) | LangGraph [persistence](https://docs.langchain.com/oss/python/langgraph/persistence) via `checkpointer=...` |


Read the map two ways. Going *right*, `deepagents` is a genuine gift: planning, a virtual filesystem, automatic summarization, sub-agents, and human-in-the-loop arrive already assembled, tested, and maintained by people who do this full time. You skip the wiring and inherit sensible defaults. For a real product under deadline, that is often the correct call, and nothing you learned is wasted — you can read its middleware stack precisely *because* you built each layer once.

Going *left* is the cost, and it is real. Our loop is thirty lines you can read; its graph is a stack of middleware whose ordering and hidden prompts you must learn to reason about. Our gate is a rule you wrote; its `interrupt_on` is a config surface you configure. Our `Budget` had no home at all — the one guardrail the framework does not ship, which is telling, because cost ceilings are exactly where a generic harness cannot guess your limits.

And a subtle one: our every model call ran through `shoplab.llm.complete`, so the cost `LEDGER` and Phoenix saw all of them; the moment we handed control to `ChatOpenAI`, that seam moved. Transparency and control are not free features you keep by default — they are things you trade away for the assembly, and getting them back inside a framework is its own project. Same ideas, packaged; know what the packaging costs before you buy it.

## Recap

| Concept | One-liner |
|---|---|
| Deep agent | a long-horizon loop that plans, uses tools, delegates, and stays inside guardrails — composed from parts, not one trick. |
| Composition over the queue | `run_agent` + `research_policy` firewall + verified gate + `Budget`, assembled per ticket from parts built in Parts 1-3. |
| `run_subagent` firewall | a sub-agent answers in one line; policy texts and its transcript never enter the parent's context. |
| Verified approval gate | `require_approval` whose approver is `rules.decide` — money moves only when the engine agrees on the exact amount. |
| Grading vs auditing | `score_ticket` grades against a labeled set; `check_decision` audits any ticket in production with no label. |
| `create_deep_agent` | `deepagents` packages planning, filesystem, sub-agent, and summarization middleware into one compiled LangGraph graph. |
| Machinery map | every hand-built piece has a home in `deepagents` — usually a middleware; the difference is packaging, not ideas. |
| The trade | a framework gives assembled, maintained machinery and takes transparency, control, and your custom seams (like `Budget`). |

## Exercises

1. Add a tool to both agents. Give the desk a `check_inventory` lookup (it is already in `standard_tools`) so the agent can note when a returned item is low on stock, then add the same capability to the `deepagents` version as a plain callable. Does either agent actually *use* it on the queue, and does adding a tool change how often the others fire? The lesson is that a toolset is a design surface in both builds — the framework does not decide relevance for you.
2. Make our deep agent survive a batch of ten. Extend `QUEUE` to ten dev tickets and watch the per-ticket `peak` tokens from `ContextLedger`. Then have `clear_ticket` call `context.compact` on its message list whenever the count crosses a threshold, and confirm the decisions still match gold. Where does compaction help, and where does it risk dropping a fact the agent still needed?
3. Find where `deepagents` keeps its plan. Give the agent a task gnarly enough to trigger planning ("triage these three tickets in order, tracking each"), invoke it, and inspect the returned state for a `todos` channel — the `write_todos` tool of `TodoListMiddleware`. Compare what it stores to the ad-hoc plan our agent carries in its system prompt: which is easier to inspect mid-run, and which is easier to change?

**Next up:** chapter 12 takes the long run and makes it *pausable* — LangGraph interrupts and a checkpointer that lets a triage stop for human input and resume from exactly where it left off, the durable cousin of the `Checkpoint` you built in chapter 08.